# 16 - Budget Optimization

## Objective

Use Marketing Mix Modeling outputs to recommend an optimal marketing budget allocation.

In a real MMM project, optimization is performed on top of the response curves
(adstock + saturation + fitted model).

This notebook demonstrates the workflow using constrained optimization.

### Business Questions

- How should a fixed marketing budget be allocated?
- Which channels deserve more investment?
- What sales uplift is expected?


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize

ROOT = Path.cwd()

channels=[
    "Google Search",
    "Meta",
    "TV",
    "YouTube",
    "Email"
]

current_spend=np.array([
    2_500_000,
    2_000_000,
    3_000_000,
    1_500_000,
    1_000_000
],dtype=float)

response=np.array([
    3.8,
    3.2,
    2.5,
    2.8,
    4.2
])

TOTAL_BUDGET=current_spend.sum()

print("Total Budget: ₹{:,.0f}".format(TOTAL_BUDGET))


## 1. Current Budget

In [ ]:

current=pd.DataFrame({
    "Channel":channels,
    "Current Spend":current_spend,
    "Estimated ROAS":response
})

display(current)


## 2. Response Function

In [ ]:

def objective(x):
    # negative because scipy minimizes
    return -np.sum(response*np.sqrt(x))


## 3. Constraints

In [ ]:

constraints=[
    {
        "type":"eq",
        "fun":lambda x: np.sum(x)-TOTAL_BUDGET
    }
]

bounds=[
    (0.5*s,1.5*s)
    for s in current_spend
]

bounds


## 4. Optimize Budget

In [ ]:

result=minimize(
    objective,
    x0=current_spend,
    bounds=bounds,
    constraints=constraints,
    method="SLSQP"
)

optimized=result.x

optimized_df=pd.DataFrame({
    "Channel":channels,
    "Current Spend":current_spend,
    "Optimized Spend":optimized
})

optimized_df["Difference"]=(
    optimized_df["Optimized Spend"]-
    optimized_df["Current Spend"]
)

display(optimized_df)


## 5. Estimated Sales Uplift

In [ ]:

current_response=np.sum(response*np.sqrt(current_spend))
optimized_response=np.sum(response*np.sqrt(optimized))

summary=pd.DataFrame({
    "Metric":[
        "Current Response",
        "Optimized Response",
        "Estimated Improvement (%)"
    ],
    "Value":[
        current_response,
        optimized_response,
        (optimized_response-current_response)/
        current_response*100
    ]
})

display(summary)


## 6. Budget Comparison

In [ ]:

x=np.arange(len(channels))
width=0.35

plt.figure(figsize=(10,5))

plt.bar(x-width/2,current_spend,width,label="Current")
plt.bar(x+width/2,optimized,width,label="Optimized")

plt.xticks(x,channels,rotation=15)
plt.ylabel("Budget (₹)")
plt.title("Budget Allocation")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Spend Share

In [ ]:

share=pd.DataFrame({
    "Channel":channels,
    "Current %":100*current_spend/current_spend.sum(),
    "Optimized %":100*optimized/optimized.sum()
})

display(share)


## 8. Scenario Planning

In [ ]:

budgets=[
    5_000_000,
    10_000_000,
    20_000_000
]

scenario=[]

for b in budgets:
    factor=b/TOTAL_BUDGET
    est=np.sum(response*np.sqrt(current_spend*factor))
    scenario.append([b,est])

scenario=pd.DataFrame(
    scenario,
    columns=[
        "Budget",
        "Estimated Response"
    ]
)

display(scenario)


# Executive Recommendations

Example observations:

- Increase investment in channels with consistently high response.
- Reduce spend in low-return channels.
- Maintain budget constraints.
- Validate optimization using business rules before deployment.

## Typical Production Enhancements

- Optimize using fitted Hill response curves
- Multi-period optimization
- Regional optimization
- Product-level optimization
- Campaign constraints
- Minimum reach constraints
- Integer programming
- Bayesian optimization

## Interview Questions

1. Why use constrained optimization?
2. Why are bounds important?
3. Why doesn't the optimizer allocate everything to one channel?
4. How would you incorporate diminishing returns?
5. How would you optimize for multiple objectives?

## Next Notebook

**17_Final_Business_Report.ipynb**

We'll create an executive-ready report summarizing:
- Data quality
- Model performance
- Channel contribution
- ROI
- Budget recommendations
- Key business insights
